# ALE3DCNN Decoder + Text-To-Brain Generation

This notebook focuses on the decoder side of the atlas-free CNN pipeline.

It can run before the global-context sweep is finished. If a best global contrastive run is available, set `ALE_CNN_BEST_GLOBAL_RUN` to that folder; otherwise it uses the known plain CNN defaults.

Stages:
1. Build extra atlas-free ALE caches for `kernel_fwhm_mm = 12` and `15`.
2. Train sparse-aware CNN autoencoders/decoders for FWHM 9/12/15.
3. Train `text -> SPECTER -> CNN text projection -> frozen CNN decoder -> generated ALE volume`.
4. Save reconstruction/generation metrics and comparison CSVs.

In [ ]:
import os, sys, time, json, platform, subprocess, shutil
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nilearn', 'nibabel', 'huggingface-hub', 'safetensors', 'adapters', 'transformers', 'pyarrow', 'matplotlib', 'pandas', 'scikit-learn', 'tqdm', 'umap-learn'])
REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')
REPO_DIR = os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn')
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[viz,notebook,metrics]'])
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = '/content/drive/MyDrive/neurovlm'
RUNS_DIR = f'{DRIVE_ROOT}/runs_ale_3dcnn_decoder_text_to_brain'
GLOBAL_RUNS_DIR = f'{DRIVE_ROOT}/runs_ale_3dcnn_best_global'
LOCAL_CACHE_DIR = '/content/ale_caches_ale_3dcnn'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
for d in [RUNS_DIR, GLOBAL_RUNS_DIR, LOCAL_CACHE_DIR, EVAL_RESOURCE_DIR]:
    os.makedirs(d, exist_ok=True)

repo_network_resource_dir = Path(REPO_DIR) / 'experiments' / 'evaluation_resources' / 'networks_labels'
drive_network_resource_dir = Path(EVAL_RESOURCE_DIR) / 'networks_labels'
drive_network_resource_dir.mkdir(parents=True, exist_ok=True)
for resource_name in ['network_test_set_labels.csv', 'network_terms_with_definitions.csv', 'network_terms_with_definitions.json']:
    src = repo_network_resource_dir / resource_name
    if src.exists():
        shutil.copy2(src, drive_network_resource_dir / resource_name)
        print('Synced evaluation resource:', drive_network_resource_dir / resource_name)

FWHM_VALUES = [float(x) for x in os.environ.get('ALE_DECODER_FWHM_VALUES', '9,12,15').split(',') if x]
AE_EPOCHS = int(os.environ.get('ALE_DECODER_AE_EPOCHS', '150'))
TEXT_DECODER_EPOCHS = int(os.environ.get('ALE_TEXT_DECODER_EPOCHS', '120'))
SKIP_COMPLETED = os.environ.get('ALE_DECODER_SKIP_COMPLETED', '1') != '0'
print('FWHM_VALUES:', FWHM_VALUES)


In [ ]:
def run_command(cmd, done_file=None, log_file=None):
    if done_file is not None and SKIP_COMPLETED and Path(done_file).exists():
        print('Skipping completed:', done_file)
        return 0
    print(cmd)
    if log_file is None:
        log_file = Path(RUNS_DIR) / f'command_{RUN_STAMP}.log'
    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with log_file.open('w') as f:
        f.write(cmd + '\n\n')
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            f.write(line)
        code = proc.wait()
        f.write(f'\nEXIT_CODE={code}\n')
    if code != 0:
        print(f'Command failed with exit code {code}. Full log: {log_file}')
        try:
            lines = log_file.read_text().splitlines()
            print('--- log tail ---')
            print('\n'.join(lines[-80:]))
            print('--- end log tail ---')
        except Exception as exc:
            print('Could not read log tail:', exc)
    return code

def fwhm_label(fwhm):
    return str(float(fwhm)).replace('.', 'p')

def cache_file_for_fwhm(fwhm):
    return f'{LOCAL_CACHE_DIR}/atlas_free_ale_4mm_fwhm{fwhm_label(fwhm)}_crop_float16.pt'

def find_best_global_run():
    env = os.environ.get('ALE_CNN_BEST_GLOBAL_RUN', '')
    if env and Path(env).exists():
        return Path(env)
    candidates = []
    for summary in Path(GLOBAL_RUNS_DIR).glob('*/main_comparison_summary_row.json'):
        try:
            row = json.loads(summary.read_text())
            score = row.get('macro_retrieval_normalized_k_recall_curve_auc') or row.get('semantic_normalized_k_recall_curve_auc') or row.get('exact_pmid_normalized_k_recall_curve_auc') or 0.0
            candidates.append((float(score), summary.parent))
        except Exception:
            pass
    if candidates:
        return sorted(candidates, reverse=True)[0][1]
    return None

BEST_GLOBAL_RUN = find_best_global_run()
print('BEST_GLOBAL_RUN:', BEST_GLOBAL_RUN)
BEST_TEXT_PROJ_CHECKPOINT = str(BEST_GLOBAL_RUN / 'checkpoints' / 'best_ale_cnn.pt') if BEST_GLOBAL_RUN else ''
print('BEST_TEXT_PROJ_CHECKPOINT:', BEST_TEXT_PROJ_CHECKPOINT or '<none>')


In [ ]:
# Build FWHM 12 and 15 caches into /content/ale_caches_ale_3dcnn. FWHM 9 is built by the regular global notebook.
cmd = (
    'python experiments/build_ale_fwhm_caches.py '
    f'--cache-dir {LOCAL_CACHE_DIR} --fwhm-values 12,15 '
    '--resolution-mm 4 --cache-dtype float16 --mode atlas_free'
)
code = run_command(cmd, done_file=Path(cache_file_for_fwhm(15)))
if code != 0:
    raise RuntimeError('FWHM cache build failed')


In [ ]:
def autoencoder_args(fwhm, run_dir):
    return [
        '--mode', 'atlas_free', '--model', 'ale_3dcnn',
        '--epochs', str(AE_EPOCHS),
        '--batch-size-auto', '--batch-size-candidates', os.environ.get('ALE_DECODER_AE_BATCH_CANDIDATES', '256,192,128,96,64,32,16,8,4'),
        '--lr', os.environ.get('ALE_DECODER_AE_LR', '3e-4'), '--weight-decay', '1e-4',
        '--val-interval', '5', '--early-stopping-patience', '25',
        '--base-channels', '48', '--num-blocks', '4', '--latent-dim', '384',
        '--dropout', '0.1', '--norm', 'group',
        '--kernel-fwhm-mm', str(fwhm), '--resolution-mm', '4', '--cache-dtype', 'float16',
        '--cache-file', cache_file_for_fwhm(fwhm),
        '--lambda-recon', '1.0', '--lambda-dice', '0.5', '--lambda-topk', '0.5', '--lambda-corr', '0.25',
        '--recon-alpha', '10.0', '--recon-gamma', '1.0',
        '--device', 'cuda' if torch.cuda.is_available() else 'auto',
        '--num-workers', '0',
        '--run-dir', str(run_dir), '--checkpoint-dir', str(run_dir / 'checkpoints'),
    ]

def train_autoencoder_for_fwhm(fwhm):
    run_dir = Path(RUNS_DIR) / f'autoencoder_plain48_fwhm{fwhm_label(fwhm)}_{RUN_STAMP}'
    best_ckpt = run_dir / 'checkpoints' / 'best_cnn_autoencoder.pt'
    cmd = 'python experiments/train_ale_cnn_autoencoder.py ' + ' '.join(map(str, autoencoder_args(fwhm, run_dir)))
    code = run_command(cmd, done_file=best_ckpt, log_file=run_dir / 'autoencoder_train.log')
    if code != 0:
        raise RuntimeError(f'Autoencoder training failed for fwhm={fwhm}')
    return str(best_ckpt), str(run_dir)

AE_RUNS = []
for fwhm in FWHM_VALUES:
    ckpt, run_dir = train_autoencoder_for_fwhm(fwhm)
    AE_RUNS.append({'fwhm': fwhm, 'autoencoder_checkpoint': ckpt, 'autoencoder_run_dir': run_dir})
AE_RUNS


In [ ]:
def text_decoder_args(item, init_name, run_dir):
    args = [
        '--mode', 'atlas_free', '--epochs', str(TEXT_DECODER_EPOCHS),
        '--batch-size-auto', '--batch-size-candidates', os.environ.get('ALE_TEXT_DECODER_BATCH_CANDIDATES', '512,384,256,192,128,96,64,32,16,8,4'),
        '--lr', os.environ.get('ALE_TEXT_DECODER_LR', '1e-4'), '--weight-decay', '1e-4',
        '--val-interval', '5', '--early-stopping-patience', '25',
        '--autoencoder-checkpoint', item['autoencoder_checkpoint'],
        '--text-proj-init', 'pretrained_infonce',
        '--kernel-fwhm-mm', str(item['fwhm']), '--resolution-mm', '4', '--cache-dtype', 'float16',
        '--cache-file', cache_file_for_fwhm(item['fwhm']),
        '--lambda-latent', '1.0', '--lambda-recon', '1.0', '--lambda-dice', '0.5', '--lambda-topk', '0.5', '--lambda-corr', '0.25',
        '--device', 'cuda' if torch.cuda.is_available() else 'auto', '--num-workers', '0',
        '--run-dir', str(run_dir), '--checkpoint-dir', str(run_dir / 'checkpoints'),
    ]
    if init_name == 'contrastive_text' and BEST_TEXT_PROJ_CHECKPOINT and Path(BEST_TEXT_PROJ_CHECKPOINT).exists():
        args.extend(['--text-proj-checkpoint', BEST_TEXT_PROJ_CHECKPOINT])
    return args

TEXT_DECODER_RUNS = []
for item in AE_RUNS:
    for init_name in ['pretrained_infonce', 'contrastive_text']:
        if init_name == 'contrastive_text' and not (BEST_TEXT_PROJ_CHECKPOINT and Path(BEST_TEXT_PROJ_CHECKPOINT).exists()):
            print('Skipping contrastive_text init; no completed best global checkpoint yet.')
            continue
        run_dir = Path(RUNS_DIR) / f'text_to_brain_{init_name}_fwhm{fwhm_label(item["fwhm"])}_{RUN_STAMP}'
        cmd = 'python experiments/train_ale_text_to_brain_decoder.py ' + ' '.join(map(str, text_decoder_args(item, init_name, run_dir)))
        done = run_dir / 'text_to_brain_generation_metrics.json'
        code = run_command(cmd, done_file=done, log_file=run_dir / 'text_to_brain_train.log')
        if code != 0:
            raise RuntimeError(f'Text-to-brain training failed: {init_name} fwhm={item["fwhm"]}')
        TEXT_DECODER_RUNS.append({'fwhm': item['fwhm'], 'init': init_name, 'run_dir': str(run_dir)})
TEXT_DECODER_RUNS


In [ ]:
import pandas as pd
rows = []
for payload in TEXT_DECODER_RUNS:
    metrics_path = Path(payload['run_dir']) / 'text_to_brain_generation_metrics.json'
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text())
    for split in ['val', 'test']:
        row = {'fwhm': payload['fwhm'], 'init': payload['init'], 'split': split, 'run_dir': payload['run_dir']}
        row.update(metrics.get(split, {}))
        rows.append(row)
summary = pd.DataFrame(rows)
summary_path = Path(RUNS_DIR) / f'text_to_brain_decoder_summary_{RUN_STAMP}.csv'
summary.to_csv(summary_path, index=False)
print('summary:', summary_path)
display(summary.sort_values(['split', 'top5_dice', 'spatial_corr'], ascending=[True, False, False]))
